# DP::Ph3 averaged voltage source inverter, unbalanced validation

Validates `DP::Ph3::AvVoltSourceInverterStateSpace` against an EMT reference
under a single-line-to-ground fault, which a single-phase dynamic phasor model
cannot represent. Both domains run off one shared power flow. A balanced
three-phase fault sets the per-phase tracking floor; a moderate phase-A fault
is the unbalanced case. Both close at 0.1 s and hold to the end.

Dynamic phasor three-phase quantities use the RMS three-phase scale, EMT uses
peak phase, so `dpsimpy.RMS3PH_TO_PEAK1PH` is applied when comparing across domains.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path

import dpsimpy

sp = dpsimpy.sp
emt = dpsimpy.emt
dp = dpsimpy.dp

sp_ph1 = sp.ph1
emt_ph3 = emt.ph3
dp_ph3 = dp.ph3

Simulation = dpsimpy.Simulation
SystemTopology = dpsimpy.SystemTopology
Logger = dpsimpy.Logger
Domain = dpsimpy.Domain
PhaseType = dpsimpy.PhaseType
PowerflowBusType = dpsimpy.PowerflowBusType
Solver = dpsimpy.Solver

Circuit, parameters and time base match the single-phase notebook. The fault
branch closed-resistance matrix selects which phases conduct. A moderate fault
resistance keeps the phase-A sag near 0.8 pu, so the double-frequency term the
unbalance induces stays second order and the positive-sequence controller
still tracks.

In [ ]:
time_step = 100e-6
final_time = 1.0
fault_time = 0.1

system_frequency = 50.0
system_omega = 2.0 * np.pi * system_frequency

grid_voltage_rms_ll = 400.0

line_resistance = 0.3
line_inductance = 0.1e-3
line_capacitance = 1e-6
line_conductance = 1e-6

fault_resistance = 1.0
switch_open_resistance = 1e12

lf = 2e-3
cf = 10e-6
rf = 0.2
rc = 0.2

kp_pll = 0.25
ki_pll = 0.2
omega_cutoff = system_omega

p_ref = 10000.0
q_ref = 5000.0

kp_power_ctrl = 0.05
ki_power_ctrl = 0.2

kp_curr_ctrl = 0.25
ki_curr_ctrl = 1.0

sim_name_pf = "DP_Ph3_AvVoltSourceInverterStateSpace_PF"

In [ ]:
def three_phase_param(value):
    return dpsimpy.Math.single_phase_parameter_to_three_phase(value)


def three_phase_variable(value):
    return dpsimpy.Math.single_phase_variable_to_three_phase(value)


def open_resistance_matrix():
    return three_phase_param(switch_open_resistance)


def balanced_closed_matrix():
    # All three phases fault through the same resistance.
    matrix = three_phase_param(switch_open_resistance)
    for phase in range(3):
        matrix[phase, phase] = fault_resistance
    return matrix


def slg_closed_matrix():
    # Only phase A conducts; phases B and C keep the open resistance.
    matrix = three_phase_param(switch_open_resistance)
    matrix[0, 0] = fault_resistance
    return matrix


def pcc_power_from_filter_power_reference(p_filter_ref, q_filter_ref):
    v_pcc_peak_phase = dpsimpy.RMS3PH_TO_PEAK1PH * grid_voltage_rms_ll

    if abs(rc) < 1e-12 or v_pcc_peak_phase < 1e-9:
        return p_filter_ref, q_filter_ref

    # With peak phase phasors:
    #
    # P_filter = P_pcc + Rc / (1.5 * |U|^2) * (P_pcc^2 + Q_pcc^2)
    # Q_filter = Q_pcc
    q_pcc_ref = q_filter_ref
    a = rc / (1.5 * v_pcc_peak_phase**2)

    discriminant = 1.0 + 4.0 * a * (p_filter_ref - a * q_pcc_ref**2)
    if discriminant < 0.0:
        raise RuntimeError(
            "No feasible PCC power found for the requested filter-side power "
            "reference, Rc, and PCC voltage estimate."
        )

    sqrt_disc = np.sqrt(discriminant)
    p1 = (-1.0 + sqrt_disc) / (2.0 * a)
    p2 = (-1.0 - sqrt_disc) / (2.0 * a)

    p_pcc_ref = p1 if abs(p1 - p_filter_ref) < abs(p2 - p_filter_ref) else p2

    return p_pcc_ref, q_pcc_ref

In [ ]:
def time_vector(df):
    return df.iloc[:, 0].to_numpy()


def scalar_signal(df, name):
    return df[name].to_numpy()


def complex_phase(df, name, phase):
    return (df[f"{name}_{phase}.re"] + 1j * df[f"{name}_{phase}.im"]).to_numpy()


def emt_phase(df, name, phase):
    return df[f"{name}_{phase}"].to_numpy()


def rms(values):
    return float(np.sqrt(np.mean(np.asarray(values) ** 2)))


def window(time, start, end):
    return (time >= start) & (time <= end)


def reconstruct_to_emt(envelope, time, scale):
    # Shift a dynamic phasor envelope back up to an instantaneous EMT-scale
    # waveform. Equivalent to villas frequency_shift with the domain scale
    # applied.
    return scale * (
        envelope.real * np.cos(system_omega * time)
        - envelope.imag * np.sin(system_omega * time)
    )


def emt_fundamental_envelope(values, time):
    # Peak-phase fundamental complex envelope of an EMT abc column, via a
    # one-period moving average that rejects the double-frequency ripple.
    dt = time[1] - time[0]
    samples_per_period = int(round((1.0 / system_frequency) / dt))
    demodulated = 2.0 * (values * np.exp(-1j * system_omega * time))
    kernel = np.ones(samples_per_period) / samples_per_period
    return np.convolve(demodulated, kernel, mode="same"), samples_per_period

In [ ]:
def run_power_flow():
    Logger.set_log_dir("logs/" + sim_name_pf)

    n_grid = sp.SimNode("nGrid", PhaseType.Single)
    n_pcc = sp.SimNode("nPcc", PhaseType.Single)

    slack = sp_ph1.NetworkInjection("Slack")
    slack.set_parameters(grid_voltage_rms_ll)
    slack.set_base_voltage(grid_voltage_rms_ll)
    slack.modify_power_flow_bus_type(PowerflowBusType.VD)

    line = sp_ph1.PiLine("Line")
    line.set_parameters(
        line_resistance,
        line_inductance,
        line_capacitance,
        line_conductance,
    )
    line.set_base_voltage(grid_voltage_rms_ll)

    p_pcc_ref, q_pcc_ref = pcc_power_from_filter_power_reference(p_ref, q_ref)

    # Negative load = injected P/Q at PCC. The fault is a timed event and is not
    # part of the balanced power flow.
    inverter_injection = sp_ph1.Load("INV_SSN_PLL")
    inverter_injection.set_parameters(-p_pcc_ref, -q_pcc_ref, grid_voltage_rms_ll)
    inverter_injection.modify_power_flow_bus_type(PowerflowBusType.PQ)

    slack.connect([n_grid])
    line.connect([n_grid, n_pcc])
    inverter_injection.connect([n_pcc])

    system = SystemTopology(
        system_frequency,
        [n_grid, n_pcc],
        [slack, line, inverter_injection],
    )

    logger = Logger(sim_name_pf)
    logger.log_attribute("v_grid_pf", n_grid.attr("v"))
    logger.log_attribute("v_pcc_pf", n_pcc.attr("v"))

    time_step_pf = final_time
    final_time_pf = final_time + time_step_pf

    sim = Simulation(sim_name_pf)
    sim.set_system(system)
    sim.set_time_step(time_step_pf)
    sim.set_final_time(final_time_pf)
    sim.set_domain(Domain.SP)
    sim.set_solver(Solver.NRP)
    sim.set_solver_component_behaviour(dpsimpy.SolverBehaviour.Initialization)
    sim.do_init_from_nodes_and_terminals(False)
    sim.add_logger(logger)
    sim.run()

    return system


system_pf = run_power_flow()

In [ ]:
def set_inverter_parameters(inverter):
    inverter.set_parameters(
        lf,
        cf,
        rf,
        rc,
        system_omega,
        kp_pll,
        ki_pll,
        omega_cutoff,
        p_ref,
        q_ref,
        kp_power_ctrl,
        ki_power_ctrl,
        kp_curr_ctrl,
        ki_curr_ctrl,
    )


def add_inverter_logs(logger, node_pcc, inverter, fault):
    logger.log_attribute("v_pcc", node_pcc.attr("v"))
    logger.log_attribute("i_inv", inverter.attr("i_intf"))
    logger.log_attribute("i_fault", fault.attr("i_intf"))
    logger.log_attribute("vc_d", inverter.attr("vc_d"))
    logger.log_attribute("vc_q", inverter.attr("vc_q"))
    logger.log_attribute("p_inst", inverter.attr("p_inst"))
    logger.log_attribute("q_inst", inverter.attr("q_inst"))
    logger.log_attribute("omega_pll", inverter.attr("omega_pll"))


def run_emt_simulation(system_pf, closed_matrix, sim_name):
    Logger.set_log_dir("logs/" + sim_name)

    n_grid = emt.SimNode("nGrid", PhaseType.ABC)
    n_pcc = emt.SimNode("nPcc", PhaseType.ABC)

    slack = emt_ph3.NetworkInjection("Slack")
    slack.set_parameters(three_phase_variable(grid_voltage_rms_ll), system_frequency)

    line = emt_ph3.PiLine("Line")
    line.set_parameters(
        three_phase_param(line_resistance),
        three_phase_param(line_inductance),
        three_phase_param(line_capacitance),
        three_phase_param(line_conductance),
    )

    inverter = emt_ph3.AvVoltSourceInverterStateSpace("INV_SSN_PLL")
    set_inverter_parameters(inverter)

    fault = emt_ph3.Switch("Fault")
    fault.set_parameters(open_resistance_matrix(), closed_matrix, False)

    slack.connect([n_grid])
    line.connect([n_grid, n_pcc])

    # terminal 0 = GND, terminal 1 = nPcc
    inverter.connect([emt.SimNode.gnd, n_pcc])

    # The fault branch shunts the PCC to ground; closing it at fault_time
    # creates the disturbance.
    fault.connect([n_pcc, emt.SimNode.gnd])

    system = SystemTopology(
        system_frequency,
        [n_grid, n_pcc],
        [slack, line, inverter, fault],
    )

    system.init_with_powerflow(system_pf, Domain.EMT)

    logger = Logger(sim_name)
    add_inverter_logs(logger, n_pcc, inverter, fault)

    sim = Simulation(sim_name)
    sim.set_system(system)
    sim.add_logger(logger)
    sim.set_domain(Domain.EMT)
    sim.set_solver(Solver.MNA)
    sim.do_system_matrix_recomputation(True)
    sim.do_init_from_nodes_and_terminals(True)

    sim.add_event(dpsimpy.event.SwitchEvent3Ph(fault_time, fault, True))

    sim.set_time_step(time_step)
    sim.set_final_time(final_time)
    sim.run()


def run_dp_simulation(system_pf, closed_matrix, sim_name):
    Logger.set_log_dir("logs/" + sim_name)

    n_grid = dp.SimNode("nGrid", PhaseType.ABC)
    n_pcc = dp.SimNode("nPcc", PhaseType.ABC)

    slack = dp_ph3.NetworkInjection("Slack")

    line = dp_ph3.PiLine("Line")
    line.set_parameters(
        three_phase_param(line_resistance),
        three_phase_param(line_inductance),
        three_phase_param(line_capacitance),
        three_phase_param(line_conductance),
    )

    inverter = dp_ph3.AvVoltSourceInverterStateSpace("INV_SSN_PLL")
    set_inverter_parameters(inverter)

    fault = dp_ph3.Switch("Fault")
    fault.set_parameters(open_resistance_matrix(), closed_matrix, False)

    slack.connect([n_grid])
    line.connect([n_grid, n_pcc])

    # terminal 0 = GND, terminal 1 = nPcc
    inverter.connect([dp.SimNode.gnd, n_pcc])

    fault.connect([n_pcc, dp.SimNode.gnd])

    system = SystemTopology(
        system_frequency,
        [n_grid, n_pcc],
        [slack, line, inverter, fault],
    )

    system.init_with_powerflow(system_pf, Domain.DP)

    logger = Logger(sim_name)
    add_inverter_logs(logger, n_pcc, inverter, fault)

    sim = Simulation(sim_name)
    sim.set_system(system)
    sim.add_logger(logger)
    sim.set_domain(Domain.DP)
    sim.set_solver(Solver.MNA)
    sim.do_system_matrix_recomputation(True)
    sim.do_init_from_nodes_and_terminals(True)

    sim.add_event(dpsimpy.event.SwitchEvent3Ph(fault_time, fault, True))

    sim.set_time_step(time_step)
    sim.set_final_time(final_time)
    sim.run()

In [ ]:
sim_name_emt_balanced = "DP_Ph3_AvVoltSourceInverterStateSpace_EMT_balanced"
sim_name_dp_balanced = "DP_Ph3_AvVoltSourceInverterStateSpace_DP_balanced"
sim_name_emt_slg = "DP_Ph3_AvVoltSourceInverterStateSpace_EMT_slg"
sim_name_dp_slg = "DP_Ph3_AvVoltSourceInverterStateSpace_DP_slg"

run_emt_simulation(system_pf, balanced_closed_matrix(), sim_name_emt_balanced)
run_dp_simulation(system_pf, balanced_closed_matrix(), sim_name_dp_balanced)
run_emt_simulation(system_pf, slg_closed_matrix(), sim_name_emt_slg)
run_dp_simulation(system_pf, slg_closed_matrix(), sim_name_dp_slg)

In [ ]:
def read_result(sim_name):
    df = pd.read_csv(
        Path("logs") / sim_name / f"{sim_name}.csv",
        skipinitialspace=True,
    )
    df.columns = [c.strip() for c in df.columns]
    return df


emt_balanced = read_result(sim_name_emt_balanced)
dp_balanced = read_result(sim_name_dp_balanced)
emt_slg = read_result(sim_name_emt_slg)
dp_slg = read_result(sim_name_dp_slg)

time = time_vector(dp_slg)

dt = time[1] - time[0]
samples_per_period = int(round((1.0 / system_frequency) / dt))
margin = samples_per_period * dt

pre_window = window(time, 0.02, fault_time - 0.01)
post_window = window(time, fault_time + 0.5, final_time)
# Interior post window, trimmed so the one-period demodulation is valid.
post_interior = window(time, fault_time + 0.5 + margin, final_time - margin)

assert pre_window.any() and post_window.any() and post_interior.any()

for name, df in [
    (sim_name_emt_slg, emt_slg),
    (sim_name_dp_slg, dp_slg),
    (sim_name_emt_balanced, emt_balanced),
    (sim_name_dp_balanced, dp_balanced),
]:
    assert len(df) == len(dp_slg), f"{name}: sample count differs from the DP SLG run."

print(dp_slg.head())

## Carry-over sanity checks

Parameter validation, no NaN or Inf, matching sample counts, and fault current
negligible before the event and present only on the faulted phase afterwards.

In [ ]:
def expect_parameter_exception(**overrides):
    params = dict(
        lf=lf,
        cf=cf,
        rf=rf,
        rc=rc,
        omega_n=system_omega,
        kp_pll=kp_pll,
        ki_pll=ki_pll,
        omega_cutoff=omega_cutoff,
        p_ref=p_ref,
        q_ref=q_ref,
        kp_power_ctrl=kp_power_ctrl,
        ki_power_ctrl=ki_power_ctrl,
        kp_curr_ctrl=kp_curr_ctrl,
        ki_curr_ctrl=ki_curr_ctrl,
    )
    params.update(overrides)

    inverter = dp_ph3.AvVoltSourceInverterStateSpace("InvalidParameterTest")
    try:
        inverter.set_parameters(
            params["lf"],
            params["cf"],
            params["rf"],
            params["rc"],
            params["omega_n"],
            params["kp_pll"],
            params["ki_pll"],
            params["omega_cutoff"],
            params["p_ref"],
            params["q_ref"],
            params["kp_power_ctrl"],
            params["ki_power_ctrl"],
            params["kp_curr_ctrl"],
            params["ki_curr_ctrl"],
        )
    except Exception:
        return
    raise AssertionError(f"Expected set_parameters to raise for overrides: {overrides}")


expect_parameter_exception(lf=0.0)
expect_parameter_exception(cf=0.0)
expect_parameter_exception(rf=-1.0)
expect_parameter_exception(rc=0.0)
expect_parameter_exception(omega_n=0.0)
expect_parameter_exception(ki_pll=0.0)
expect_parameter_exception(ki_power_ctrl=0.0)
expect_parameter_exception(ki_curr_ctrl=0.0)

for name, df in [("EMT", emt_slg), ("DP", dp_slg)]:
    assert not df.isna().any().any(), f"{name} SLG result contains NaN or Inf."

fault_a_dp = np.abs(complex_phase(dp_slg, "i_fault", 0))
fault_b_dp = np.abs(complex_phase(dp_slg, "i_fault", 1))
fault_c_dp = np.abs(complex_phase(dp_slg, "i_fault", 2))

pre_fault_window = window(time, 0.02, fault_time - 0.01)
post_fault_window = window(time, fault_time + 0.05, final_time)

assert (
    rms(fault_a_dp[pre_fault_window]) < 1e-2
), "Fault current present before the event."
assert (
    fault_a_dp[post_fault_window].mean() > 1.0
), "Fault current absent after the event."
assert rms(fault_b_dp[post_fault_window]) < 1e-2, "Phase B conducts in a phase-A fault."
assert rms(fault_c_dp[post_fault_window]) < 1e-2, "Phase C conducts in a phase-A fault."

print("Carry-over sanity checks passed.")

## Per-phase electrical envelopes

The capability a single-phase model cannot provide: the diagonal per-phase
filter carries the fault, so each phase tracks its own EMT phase. The balanced
fault sets the tracking floor. Each DP envelope is shifted back up to an
instantaneous waveform and compared to the EMT abc column; the faulted phase-A
fundamental is compared tightly; the per-phase magnitude spread, the headline
unbalance signature, is checked non-trivial and consistent between domains.

DP gives phase B and phase C the same magnitude, while EMT splits them with B
above C. The positive-sequence controller and the symmetric passive filter
present equal positive and negative sequence impedance, so the healthy phases
stay mirror-symmetric; EMT's dq controller re-emits the double-frequency ripple
as a negative-sequence response that separates them. Hence the looser
healthy-phase tolerance, and the DP phase-B curve sitting on top of DP phase-C
in the plot below. Resolving the split needs a dual-sequence formulation, out
of scope here.

In [ ]:
# Balanced fault: per-phase instantaneous reconstruction floor.
reconstruct_floor = 0.0
for phase in range(3):
    reconstruction = reconstruct_to_emt(
        complex_phase(dp_balanced, "v_pcc", phase), time, dpsimpy.RMS3PH_TO_PEAK1PH
    )
    reference = emt_phase(emt_balanced, "v_pcc", phase)
    reconstruct_floor = max(
        reconstruct_floor, rms((reconstruction - reference)[post_window])
    )

reconstruct_tolerance = max(3.0, 2.0 * reconstruct_floor)
print(f"Balanced per-phase reconstruction floor: {reconstruct_floor:.2f} V")

# Pre-fault (still balanced) the SLG run must track just as tightly.
for phase in range(3):
    reconstruction = reconstruct_to_emt(
        complex_phase(dp_slg, "v_pcc", phase), time, dpsimpy.RMS3PH_TO_PEAK1PH
    )
    reference = emt_phase(emt_slg, "v_pcc", phase)
    error = rms((reconstruction - reference)[pre_window])
    assert error <= reconstruct_tolerance, (
        f"Pre-fault phase {phase} reconstruction error {error:.2f} V exceeds "
        f"{reconstruct_tolerance:.2f} V."
    )

print("Pre-fault per-phase reconstruction within the balanced floor.")

In [ ]:
# Per-phase fundamental magnitude under the phase-A fault.
dp_fundamental = {
    phase: np.abs(complex_phase(dp_slg, "v_pcc", phase)) * dpsimpy.RMS3PH_TO_PEAK1PH
    for phase in range(3)
}
emt_fundamental = {
    phase: np.abs(emt_fundamental_envelope(emt_phase(emt_slg, "v_pcc", phase), time)[0])
    for phase in range(3)
}

# Balanced fundamental floor.
fundamental_floor = 0.0
for phase in range(3):
    dp_mag = (
        np.abs(complex_phase(dp_balanced, "v_pcc", phase)) * dpsimpy.RMS3PH_TO_PEAK1PH
    )
    emt_mag = np.abs(
        emt_fundamental_envelope(emt_phase(emt_balanced, "v_pcc", phase), time)[0]
    )
    fundamental_floor = max(fundamental_floor, rms((dp_mag - emt_mag)[post_interior]))

faulted_tolerance = max(4.0, 2.0 * fundamental_floor)
healthy_tolerance = max(14.0, 8.0 * fundamental_floor)
print(f"Balanced fundamental floor: {fundamental_floor:.2f} V")

faulted_error = rms((dp_fundamental[0] - emt_fundamental[0])[post_interior])
assert faulted_error <= faulted_tolerance, (
    f"Faulted phase-A fundamental error {faulted_error:.2f} V exceeds "
    f"{faulted_tolerance:.2f} V."
)
print(f"Faulted phase-A fundamental error: {faulted_error:.2f} V (tight).")

for phase in (1, 2):
    error = rms((dp_fundamental[phase] - emt_fundamental[phase])[post_interior])
    assert error <= healthy_tolerance, (
        f"Healthy phase {phase} fundamental error {error:.2f} V exceeds "
        f"{healthy_tolerance:.2f} V."
    )
    print(
        f"Healthy phase {phase} fundamental error: {error:.2f} V (documented looser)."
    )

In [ ]:
# Headline: the per-phase magnitude spread is non-trivial and agrees in both
# domains. This is the unbalance signature a single-phase model cannot produce.
dp_spread = (
    dp_fundamental[0][post_interior].mean() - dp_fundamental[1][post_interior].mean()
)
emt_spread = (
    emt_fundamental[0][post_interior].mean() - emt_fundamental[1][post_interior].mean()
)

spread_relative_difference = abs(dp_spread - emt_spread) / abs(emt_spread)

assert abs(dp_spread) > 30.0, "DP per-phase spread is trivial."
assert abs(emt_spread) > 30.0, "EMT per-phase spread is trivial."
assert spread_relative_difference <= 0.25, (
    f"Per-phase spread disagreement {spread_relative_difference * 100:.1f}% "
    "exceeds 25%."
)

print(
    f"Per-phase spread |Va|-|Vb|: DP={dp_spread:.1f} V, EMT={emt_spread:.1f} V "
    f"({spread_relative_difference * 100:.1f}% difference)."
)
print("Per-phase electrical envelope checks passed.")

## Positive-sequence controller scalars

The single-dq-frame positive-sequence controller does not represent the
double-frequency component unbalance induces, so under the fault the EMT
scalars carry a strong double-frequency ripple while the DP scalars are smooth
and track its average. They are compared against a one-period moving average of
the EMT scalar with a loosened tolerance. A bolted fault would diverge further;
this validates the per-phase electrical model, not negative-sequence ride
through. The pre-fault window is still balanced and stays tight. `vc_d`/`vc_q`
are voltages and take the scale factor; `p_inst`/`q_inst`/`omega_pll` do not.

In [ ]:
kernel = np.ones(samples_per_period) / samples_per_period


def moving_average(values):
    return np.convolve(values, kernel, mode="same")


def check_scalar(name, scale, pre_tol, post_tol):
    dp_values = scalar_signal(dp_slg, name) * scale
    emt_values = scalar_signal(emt_slg, name)
    emt_average = moving_average(emt_values)

    pre_error = rms((dp_values - emt_values)[pre_window])
    post_error = rms((dp_values - emt_average)[post_interior])

    assert (
        pre_error <= pre_tol
    ), f"Pre-fault {name} error {pre_error:.4f} exceeds {pre_tol}."
    assert (
        post_error <= post_tol
    ), f"Post-fault {name} error {post_error:.4f} exceeds {post_tol}."
    print(
        f"{name}: pre={pre_error:.4f}, post={post_error:.4f} (limits {pre_tol}, {post_tol})."
    )


# p_inst tracks the average within ~1.5%; q_inst, omega_pll, vc_q, vc_d are tight.
check_scalar("p_inst", 1.0, 0.02 * abs(p_ref), 0.03 * abs(p_ref))
check_scalar("q_inst", 1.0, 0.02 * abs(q_ref), 0.02 * abs(q_ref))
check_scalar("omega_pll", 1.0, 1e-2, 1e-2)
check_scalar("vc_q", dpsimpy.RMS3PH_TO_PEAK1PH, 0.5, 0.5)
check_scalar("vc_d", dpsimpy.RMS3PH_TO_PEAK1PH, 10.0, 10.0)

print("Positive-sequence controller scalar checks passed.")

## Plots

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(10, 9), sharex=True)
phase_labels = ["A (faulted)", "B", "C"]
for phase, ax in enumerate(axes):
    reconstruction = reconstruct_to_emt(
        complex_phase(dp_slg, "v_pcc", phase), time, dpsimpy.RMS3PH_TO_PEAK1PH
    )
    ax.plot(time, emt_phase(emt_slg, "v_pcc", phase), label="EMT")
    ax.plot(time, reconstruction, label="DP reconstructed", linestyle="--")
    ax.set_ylabel(f"phase {phase_labels[phase]} [V]")
    ax.set_xlim(0.0, final_time)
    ax.grid(True)
    ax.legend(loc="upper right")
axes[-1].set_xlabel("time [s]")
axes[0].set_title("PCC voltage per phase, EMT vs DP reconstruction")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))
for phase, label in zip(range(3), ["A (faulted)", "B", "C"]):
    (line,) = plt.plot(time, dp_fundamental[phase], label=f"phase {label} (DP)")
    plt.plot(
        time,
        emt_fundamental[phase],
        linestyle="--",
        color=line.get_color(),
        label=f"phase {label} (EMT)",
    )
plt.xlim(0.0, final_time)
plt.xlabel("time [s]")
plt.ylabel("fundamental magnitude [V]")
plt.title("Per-phase PCC voltage fundamental magnitude")
plt.grid(True)
plt.legend(loc="lower left", ncol=3)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(10, 9), sharex=True)
scalar_plots = [
    ("p_inst", 1.0, "active power [W]", p_ref),
    ("q_inst", 1.0, "reactive power [var]", q_ref),
    ("omega_pll", 1.0, "PLL frequency [rad/s]", system_omega),
]
for (name, scale, ylabel, reference), ax in zip(scalar_plots, axes):
    ax.plot(time, scalar_signal(emt_slg, name), label="EMT")
    ax.plot(
        time,
        moving_average(scalar_signal(emt_slg, name)),
        label="EMT one-period average",
    )
    ax.plot(time, scalar_signal(dp_slg, name) * scale, label="DP", linestyle="--")
    ax.axhline(reference, linestyle=":", color="gray")
    ax.set_ylabel(ylabel)
    ax.set_xlim(0.0, final_time)
    ax.grid(True)
    ax.legend(loc="upper right")
axes[-1].set_xlabel("time [s]")
axes[0].set_title("Positive-sequence controller scalars under the phase-A fault")
plt.tight_layout()
plt.show()